# Phase 1 — Pipeline Check (Colab)

Proves the full chain works on a real GPU before committing hours:
**quantize Qwen → eval FP16 + INT4 on GSM8K + ARC → aggregate → analyze → tables/figures**.

Qwen2.5-3B-Instruct is **ungated**, so Phase 1 needs **no HF token**.

### Before you run
1. Upload the `research_project/` folder to your Google Drive at `MyDrive/research_project` (so results survive a session dying).
2. Runtime → Change runtime type → **GPU** (T4 is fine).
3. Run the cells top to bottom.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Mount Drive and locate the project

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/research_project'
assert os.path.isdir(PROJECT_DIR), (
    f'{PROJECT_DIR} not found. Upload the research_project/ folder to MyDrive first.')
os.chdir(PROJECT_DIR)
print('Working in:', os.getcwd())
print(os.listdir('.'))

## 3. Install dependencies

Colab already ships a CUDA build of PyTorch, so we don't reinstall it. This installs
lm-eval, the GPTQ backend, the analysis libs, and HumanEval's executor. Takes a few minutes.

In [ ]:
!pip install -q lm-eval transformers accelerate datasets sentencepiece \
    gptqmodel bitsandbytes scipy pandas numpy matplotlib seaborn pyyaml
!pip install -q human-eval  # HumanEval execution backend (used in later phases)

import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 4. (Optional) HuggingFace token

**Not needed for Phase 1** (Qwen is open). Only set this when you move to gated
models (Llama, Gemma). To use it, uncomment and paste your token.

In [ ]:
# import os
# os.environ['HF_TOKEN'] = 'hf_...'  # accept model licenses on huggingface.co first

## 5. Run Phase 1

Quantizes Qwen2.5-3B-Instruct to INT4-GPTQ (once, reused), then evaluates
FP16 and INT4 on GSM8K and ARC-Challenge. Results land in `results/raw/` (on Drive).

In [ ]:
!bash scripts/run_all_evals.sh phase1

## 6. Aggregate → analyze → tables

In [ ]:
!python scripts/aggregate_results.py --raw_dir results/raw/ --output_dir results/processed/
!python scripts/analyze_results.py --results_csv results/processed/results_table.csv --output_dir analysis/
!python analysis/make_paper_tables.py --output_dir paper/tables/

## 7. Inspect results

The decision question for Phase 1/2: **does degradation differ by capability?**
If Math/Code drop a lot while Factual barely moves, you have signal.

In [ ]:
import pandas as pd
from IPython.display import display

print('=== Absolute scores ===')
display(pd.read_csv('results/processed/results_table.csv'))

deg = 'analysis/degradation.csv'
import os
if os.path.exists(deg):
    print('=== Relative degradation (%) ===')
    display(pd.read_csv(deg))

In [ ]:
from IPython.display import Image
import os
fig = 'analysis/figures/degradation_heatmap.png'
Image(fig) if os.path.exists(fig) else print('Heatmap not found — check the analyze step above.')

## Done

Everything is saved under `MyDrive/research_project/` (raw JSON, processed CSVs,
analysis outputs, paper tables/figures), so it survives the session ending.

**Next:** if there's a clear per-capability signal, move to `phase3_open`
(all ungated models, full 6 benchmarks) — same flow, just change the phase in cell 5.